# Vauban Training v2 — Single-Stage YOLO with Real Type Names

Trains YOLOv8 to detect AND classify aircraft in satellite imagery in one shot.

**20 classes:** Su-35, C-130, C-17, C-5, F-16, Tu-160, E-3, B-52, P-3C, B-1B, E-8, Tu-22M, F-15, KC-135, F-22, F/A-18, Tu-95, KC-10, Su-34, Su-24

Output: `vauban_detect_v2.pt`

In [ ]:
!pip install -q ultralytics kagglehub scikit-learn matplotlib seaborn pyyaml

## Download Dataset

In [ ]:
import os
os.environ['KAGGLE_API_KEY'] = 'KGAT_1fa689529f5ea2d60f8dc273b79f9ac4'  # Your key

import kagglehub
mar20_path = kagglehub.dataset_download('khlaifiabilel/military-aircraft-recognition-dataset')
print(f'Dataset: {mar20_path}')

## Convert to YOLO Format with Real Type Names

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil
import random
import yaml

SRC = Path(mar20_path)
OUT = Path('datasets/mar20_v2')

CLASS_MAP = {
    'A1': 'Su-35',
    'A2': 'C-130',
    'A3': 'C-17',
    'A4': 'C-5',
    'A5': 'F-16',
    'A6': 'Tu-160',
    'A7': 'E-3',
    'A8': 'B-52',
    'A9': 'P-3C',
    'A10': 'B-1B',
    'A11': 'E-8',
    'A12': 'Tu-22M',
    'A13': 'F-15',
    'A14': 'KC-135',
    'A15': 'F-22',
    'A16': 'FA-18',
    'A17': 'Tu-95',
    'A18': 'KC-10',
    'A19': 'Su-34',
    'A20': 'Su-24',
}

CLASSES = list(dict.fromkeys(CLASS_MAP.values()))  # Unique, preserving order
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

print(f'{len(CLASSES)} classes:')
for i, c in enumerate(CLASSES):
    print(f'  {i}: {c}')

In [ ]:
def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)
    objects = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        objects.append({
            'class': obj.find('name').text,
            'xmin': int(bbox.find('xmin').text),
            'ymin': int(bbox.find('ymin').text),
            'xmax': int(bbox.find('xmax').text),
            'ymax': int(bbox.find('ymax').text),
            'img_w': w, 'img_h': h,
        })
    return objects

def voc_to_yolo(obj):
    mapped = CLASS_MAP.get(obj['class'])
    if not mapped or obj['img_w'] == 0 or obj['img_h'] == 0:
        return None
    cid = CLASS_TO_ID[mapped]
    w, h = obj['img_w'], obj['img_h']
    cx = max(0, min(1, ((obj['xmin'] + obj['xmax']) / 2) / w))
    cy = max(0, min(1, ((obj['ymin'] + obj['ymax']) / 2) / h))
    bw = max(0, min(1, (obj['xmax'] - obj['xmin']) / w))
    bh = max(0, min(1, (obj['ymax'] - obj['ymin']) / h))
    return f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'

xml_dir = SRC / 'Annotations' / 'Horizontal Bounding Boxes'
img_dir = SRC / 'JPEGImages'
train_ids = set((SRC / 'ImageSets' / 'Main' / 'train.txt').read_text().strip().split('\n'))
test_list = list(set((SRC / 'ImageSets' / 'Main' / 'test.txt').read_text().strip().split('\n')))
random.seed(42)
random.shuffle(test_list)
mid = len(test_list) // 2
val_ids, test_ids = set(test_list[:mid]), set(test_list[mid:])

splits = {'train': train_ids, 'val': val_ids, 'test': test_ids}
class_counts = {c: 0 for c in CLASSES}

for split_name, ids in splits.items():
    img_out = OUT / 'images' / split_name
    lbl_out = OUT / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    count = 0
    for img_id in sorted(ids):
        xml_path = xml_dir / f'{img_id}.xml'
        img_path = img_dir / f'{img_id}.jpg'
        if not xml_path.exists() or not img_path.exists():
            continue
        objects = parse_voc_xml(xml_path)
        lines = []
        for o in objects:
            line = voc_to_yolo(o)
            if line:
                lines.append(line)
                class_counts[CLASS_MAP[o['class']]] += 1
        if not lines:
            continue
        shutil.copy2(img_path, img_out / f'{img_id}.jpg')
        (lbl_out / f'{img_id}.txt').write_text('\n'.join(lines) + '\n')
        count += 1
    print(f'{split_name}: {count} images')

# Write data.yaml
data_yaml = {
    'path': str(OUT.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
with open(OUT / 'data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(f'\nClass distribution:')
for c, n in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f'  {c:<12} {n}')

## Train YOLOv8

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

os.makedirs('models', exist_ok=True)

model = YOLO('yolov8n.pt')

results = model.train(
    data='datasets/mar20_v2/data.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    project='runs/detection',
    name='vauban_v2',
    freeze=10,
    lr0=0.001,
    lrf=0.01,
    degrees=180.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
)

best = Path('runs/detection/vauban_v2/weights/best.pt')
if best.exists():
    shutil.copy(best, 'models/vauban_detect_v2.pt')
    print('Model saved to models/vauban_detect_v2.pt')

## Metrics — Training Curves

In [ ]:
import shutil
from IPython.display import display, Image as IPImage

os.makedirs('metrics', exist_ok=True)
run_dir = Path('runs/detection/vauban_v2')

for name in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
             'BoxF1_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxPR_curve.png',
             'labels.jpg', 'val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    src = run_dir / name
    if src.exists():
        shutil.copy(src, f'metrics/{name}')

print('=== Training Curves ===')
if (run_dir / 'results.png').exists():
    display(IPImage(filename=str(run_dir / 'results.png'), width=900))

In [ ]:
print('=== F1 vs Confidence ===')
if (run_dir / 'BoxF1_curve.png').exists():
    display(IPImage(filename=str(run_dir / 'BoxF1_curve.png'), width=700))

print('\n=== Precision-Recall Curve ===')
if (run_dir / 'BoxPR_curve.png').exists():
    display(IPImage(filename=str(run_dir / 'BoxPR_curve.png'), width=700))

In [ ]:
print('=== Confusion Matrix (Normalized) ===')
if (run_dir / 'confusion_matrix_normalized.png').exists():
    display(IPImage(filename=str(run_dir / 'confusion_matrix_normalized.png'), width=800))

In [ ]:
print('=== Validation Predictions ===')
for name in ['val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    if (run_dir / name).exists():
        display(IPImage(filename=str(run_dir / name), width=900))

## Test Set Evaluation

In [ ]:
from ultralytics import YOLO

val_model = YOLO('models/vauban_detect_v2.pt')
val_results = val_model.val(data='datasets/mar20_v2/data.yaml', split='test')

print('=== Test Set Results ===')
print(f'mAP@0.5:      {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'Precision:     {val_results.box.mp:.4f}')
print(f'Recall:        {val_results.box.mr:.4f}')
print()
print(f'{"Class":<15} {"AP@0.5":>8} {"AP@0.5:0.95":>12}')
print('-' * 37)
names = val_model.names
for i in range(len(val_results.box.ap50)):
    print(f'{names[i]:<15} {val_results.box.ap50[i]:>8.4f} {val_results.box.ap[i]:>12.4f}')

## Per-Class F1 Bar Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ap50 = val_results.box.ap50
class_names = [names[i] for i in range(len(ap50))]

sorted_idx = np.argsort(ap50)
sorted_names = [class_names[i] for i in sorted_idx]
sorted_ap = ap50[sorted_idx]

russian = {'Su-35', 'Su-34', 'Su-24', 'Tu-95', 'Tu-160', 'Tu-22M'}
colors = ['#e74c3c' if n in russian else '#2d7dd2' for n in sorted_names]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(range(len(sorted_names)), sorted_ap, color=colors)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel('AP@0.5')
ax.set_title('Per-Class AP@0.5 (Red = Russian, Blue = NATO/US)')
ax.set_xlim(0, 1)
ax.grid(True, axis='x', alpha=0.3)

for bar, ap in zip(bars, sorted_ap):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{ap:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('metrics/per_class_ap50.png', dpi=150, bbox_inches='tight')
plt.show()

## Download Model & Metrics

In [ ]:
import os

print('=== Model ===')
f = 'models/vauban_detect_v2.pt'
if os.path.exists(f):
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f} — {size_mb:.1f} MB')

print('\n=== Metrics ===')
for f in sorted(os.listdir('metrics')):
    print(f'  metrics/{f}')

try:
    from google.colab import files
    files.download('models/vauban_detect_v2.pt')
    for f in os.listdir('metrics'):
        if f.endswith('.png') or f.endswith('.jpg'):
            files.download(f'metrics/{f}')
except:
    print('\nNot on Colab — grab files manually')